<a href="https://colab.research.google.com/github/cras-lab/ML-examples/blob/main/DeepLearning_XOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 모듈을 설치한다.

In [1]:
import torch
import torch.nn as nn   # 신경망
import torch.optim as optim   #Optimizer 모듈

연산 환경을 설정한다.

In [2]:
torch.set_num_threads(1)  # CPU 스레드는 1개
torch.manual_seed(0)    # Seed 설정

가상의 데이터(XOR 문제)를 구성한다.

In [3]:
# 규칙:
# x1, x2의 부호가 같으면 1
# x1, x2의 부호가 다르면 0

base_X = torch.tensor([
    [ 1.0,  1.0],
    [ 1.0, -1.0],
    [-1.0,  1.0],
    [-1.0, -1.0],
])

base_y = torch.tensor([
    [1.0],
    [0.0],
    [0.0],
    [1.0],
])

딥러닝 훈련을 위해서는 데이터가 많아야 하므로, 복제<BR>
복제하면서 약간의 잡음을 추가해서 실습 데이터를 만든다.

<pre>
repeat(50, 1)
       │   │
       │   └─ 열 방향 1번 반복 = 그대로 둠
       └───── 행 방향 50번 반복 = 아래로 50번 복사
원래 데이터가 4개 행이니 총 200개 행이 만들어 진다.</pre>

In [4]:
X = base_X.repeat(50, 1)  # 첫번째 축(행)은 50번 반복,

앞의 5개 데이터만 출력해 보자

In [5]:
print(X[:10])

tensor([[ 1.,  1.],
        [ 1., -1.],
        [-1.,  1.],
        [-1., -1.],
        [ 1.,  1.],
        [ 1., -1.],
        [-1.,  1.],
        [-1., -1.],
        [ 1.,  1.],
        [ 1., -1.]])


자 이 값에 약간의 랜덤 노이즈를 추가해 값을 흔들자.

In [6]:
X = X + 0.15 * torch.randn(200, 2) # 200개 행의 두 축(열)에 랜덤값 합산

랜던화 된 후의 X 값을 출력해 보자.

In [7]:
print(X[:10])

tensor([[ 0.8311,  0.8271],
        [ 0.9624, -1.0651],
        [-0.8727,  1.1038],
        [-1.0474, -1.3173],
        [ 1.0483,  0.8105],
        [ 1.0525, -0.9538],
        [-0.9820,  1.1856],
        [-0.8325, -1.0371],
        [ 0.7971,  0.7456],
        [ 1.0850, -0.8810]])


y 값은 정답 레이블이니 단순히 똑 같이 50번 복제만 한다.

In [8]:
y = base_y.repeat(50, 1)

y 값도 출력해 보자.

In [9]:
display(y[:10])

tensor([[1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.]])

이제 데이터가 준비되었으니 신경망을 구축한다.<BR>
입력 뉴런 2개 → 은닉층 뉴런 8개 → 출력 뉴런 1개

In [10]:
model = nn.Sequential(
    nn.Linear(2, 8),   # 1층: 입력 2개를 은닉노드 8개로 변환
    nn.ReLU(),         # 비선형 활성화 함수
    nn.Linear(8, 1)    # 2층: 은닉노드 8개를 출력 1개로 변환
)

현재 신경망 구조를 그림으로 출력해 보자.<BR>
실선이면 양수이고, 쇄선이면 음수이다.<BR>
선이 두꺼울수록 가중치 절댓값이 크다

In [ ]:
import matplotlib.pyplot as plt # 그림 모듈 임포트

def draw_network(model):
    layer1_weight = model[0].weight.detach()
    layer2_weight = model[2].weight.detach()

    input_nodes = 2
    hidden_nodes = 8
    output_nodes = 1

    input_y = [0.7, 0.3]
    hidden_y = torch.linspace(0.9, 0.1, hidden_nodes).tolist()
    output_y = [0.5]

    input_x = 0
    hidden_x = 1
    output_x = 2

    plt.figure(figsize=(8, 6))

    # 입력층 노드
    for i in range(input_nodes):
        plt.scatter(input_x, input_y[i], s=800)
        plt.text(input_x, input_y[i], f"x{i+1}", ha="center", va="center")

    # 은닉층 노드
    for h in range(hidden_nodes):
        plt.scatter(hidden_x, hidden_y[h], s=800)
        plt.text(hidden_x, hidden_y[h], f"h{h+1}", ha="center", va="center")

    # 출력층 노드
    plt.scatter(output_x, output_y[0], s=800)
    plt.text(output_x, output_y[0], "y", ha="center", va="center")

    # 가중치 크기에 따라 선 두께 조절
    max_w1 = layer1_weight.abs().max().item()
    max_w2 = layer2_weight.abs().max().item()
    max_w = max(max_w1, max_w2)

    # 입력층 → 은닉층 연결
    for h in range(hidden_nodes):
        for i in range(input_nodes):
            w = layer1_weight[h, i].item()
            line_width = 0.5 + 3.0 * abs(w) / max_w
            line_style = "-" if w >= 0 else "--"

            plt.plot(
                [input_x, hidden_x],
                [input_y[i], hidden_y[h]],
                linewidth=line_width,
                linestyle=line_style,
                alpha=0.6,
            )

    # 은닉층 → 출력층 연결
    for h in range(hidden_nodes):
        w = layer2_weight[0, h].item()
        line_width = 0.5 + 3.0 * abs(w) / max_w
        line_style = "-" if w >= 0 else "--"

        plt.plot(
            [hidden_x, output_x],
            [hidden_y[h], output_y[0]],
            linewidth=line_width,
            linestyle=line_style,
            alpha=0.6,
        )

    plt.title("Neural Network Structure: 2 → 8 → 1")
    plt.axis("off")
    plt.show()


draw_network(model)

학습전 후 가중치 비교를 위해 학습전 가중치를 저장하자.

In [12]:
# 학습 전 가중치 저장
before_layer1_weight = model[0].weight.detach().clone()
before_layer1_bias = model[0].bias.detach().clone()
before_layer2_weight = model[2].weight.detach().clone()
before_layer2_bias = model[2].bias.detach().clone()

테스트 데이터를 준비해 보자.

In [13]:
test_X = torch.tensor([
    [ 1.0,  1.0],     # 같은 부호니까 정답은 1이다.
    [ 1.0, -1.0],     # 다른 부호니까 정답은 0이다.
    [-1.0,  1.0],     # 다른 부호니까 정답은 0이다.
    [-1.0, -1.0],     # 같은 부호니까 정답은 1이다.
])                    # 따라서 테스트 데이터의 정답은 [1,0,0,1] 이어야 한다.

학습되기 전에 예측이 어떻게 되는지 보자.

In [ ]:
with torch.no_grad():
    logits = model(test_X)
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

print("부호가 같을(=1) 확률:")
print(probs)

print("예측값:")
print(preds)

BCE(Binary Cross Entropy) 로지스틱 손실함수를 설정 <BR>
이 경우 마지막 계층에 자동으로 sigmoid 활성함수가 설정된다.(별도의 softmax 필요없음)

In [15]:
loss_fn = nn.BCEWithLogitsLoss()   # Binary

최적기를 설정한다. Adam 방식으로 학습률(lr)을 0.05로 설정<BR>
현재는 전체 모델 매개변수를 학습하며, 만약 첫번째가 freezing이고<BR>
두번째만 학습한다면 optimizer = optim.Adam(model[2].parameters(), lr=0.05) 식이 된다.

In [16]:
optimizer = optim.Adam(model.parameters(), lr=0.05)

200번 에폭을 돌며 학습을 시작한다.<BR>
20번 돌때마다 중간 과정을 출력해 본다.

In [ ]:
for epoch in range(200):
    logits = model(X)
    loss = loss_fn(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        with torch.no_grad():
            probs = torch.sigmoid(model(X))
            preds = (probs > 0.5).float()
            accuracy = (preds == y).float().mean()

        print(f"epoch {epoch + 1}, loss = {loss.item():.4f}, accuracy = {accuracy.item():.4f}")

학습이 완료 된 후 예측을 해 보자.

In [18]:
with torch.no_grad():
    logits = model(test_X)
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

print("입력:")
print(test_X)

print("확률:")
print(probs)

print("예측값:")
print(preds)

입력:
tensor([[ 1.,  1.],
        [ 1., -1.],
        [-1.,  1.],
        [-1., -1.]])
확률:
tensor([[9.9984e-01],
        [8.4927e-05],
        [5.5601e-05],
        [9.9992e-01]])
예측값:
tensor([[1.],
        [0.],
        [0.],
        [1.]])


신경망 모양을 그려 본다. <BR>
실선이면 양수이고, 쇄선이면 음수이다.<BR>
선이 두꺼울수록 가중치 절댓값이 크다

In [ ]:
def draw_network_on_axis(ax, layer1_weight, layer2_weight, title, max_w):
    input_nodes = 2
    hidden_nodes = 8
    output_nodes = 1

    input_y = [0.7, 0.3]
    hidden_y = torch.linspace(0.9, 0.1, hidden_nodes).tolist()
    output_y = [0.5]

    input_x = 0
    hidden_x = 1
    output_x = 2

    # 입력층 노드
    for i in range(input_nodes):
        ax.scatter(input_x, input_y[i], s=700)
        ax.text(input_x, input_y[i], f"x{i+1}", ha="center", va="center")

    # 은닉층 노드
    for h in range(hidden_nodes):
        ax.scatter(hidden_x, hidden_y[h], s=700)
        ax.text(hidden_x, hidden_y[h], f"h{h+1}", ha="center", va="center")

    # 출력층 노드
    ax.scatter(output_x, output_y[0], s=700)
    ax.text(output_x, output_y[0], "y", ha="center", va="center")

    # 입력층 → 은닉층
    for h in range(hidden_nodes):
        for i in range(input_nodes):
            w = layer1_weight[h, i].item()
            line_width = 0.5 + 3.0 * abs(w) / max_w
            line_style = "-" if w >= 0 else "--"

            ax.plot(
                [input_x, hidden_x],
                [input_y[i], hidden_y[h]],
                linewidth=line_width,
                linestyle=line_style,
                alpha=0.6,
            )

    # 은닉층 → 출력층
    for h in range(hidden_nodes):
        w = layer2_weight[0, h].item()
        line_width = 0.5 + 3.0 * abs(w) / max_w
        line_style = "-" if w >= 0 else "--"

        ax.plot(
            [hidden_x, output_x],
            [hidden_y[h], output_y[0]],
            linewidth=line_width,
            linestyle=line_style,
            alpha=0.6,
        )

    ax.set_title(title)
    ax.axis("off")


def draw_weight_comparison():
    after_layer1_weight = model[0].weight.detach()
    after_layer2_weight = model[2].weight.detach()

    # 전후 그래프의 선 두께 기준을 동일하게 맞춤
    max_w = max(
        before_layer1_weight.abs().max().item(),
        before_layer2_weight.abs().max().item(),
        after_layer1_weight.abs().max().item(),
        after_layer2_weight.abs().max().item(),
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    draw_network_on_axis(
        axes[0],
        before_layer1_weight,
        before_layer2_weight,
        "Before Training",
        max_w,
    )

    draw_network_on_axis(
        axes[1],
        after_layer1_weight,
        after_layer2_weight,
        "After Training",
        max_w,
    )

    plt.suptitle("Weight Comparison: Before vs After Training")
    plt.show()


draw_weight_comparison()